# UAV Formation Visualization for Leader Selection Strategies

This notebook generates qualitative trajectory visualizations and metric plots for different leader-selection strategies in the same complex obstacle environment.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from pathlib import Path
import pandas as pd

# =============================
# 0. Global Settings
# =============================

np.random.seed(7)

# Output folder.
# If you run this notebook from the repository root,
# the figures will be saved to ./figure_visualization
out_dir = Path("figure_visualization")
out_dir.mkdir(exist_ok=True)

n_uavs = 5
steps = 90
world_size = 20

start = np.array([2.0, 2.0])
goal = np.array([18.0, 18.0])

# Same complex environment for all strategies
obstacles = np.array([
    [6.0, 5.5, 1.4],
    [8.5, 8.0, 1.6],
    [11.0, 10.0, 1.5],
    [13.5, 13.0, 1.4],
    [9.5, 13.5, 1.2],
    [14.5, 7.5, 1.3],
    [5.5, 11.5, 1.2],
])

# Formation offsets around the leader/reference frame
formation_offsets = np.array([
    [0.0, 0.0],
    [-1.2, -0.8],
    [-1.2, 0.8],
    [-2.4, -0.8],
    [-2.4, 0.8],
])

strategies = ["static", "task", "communication", "perception"]

strategy_titles = {
    "static": "Static Leader",
    "task": "Task-based Leader",
    "communication": "Communication-based Leader",
    "perception": "Perception-aware Leader"
}

print("Environment and global settings loaded.")


In [ ]:

# =============================
# 1. Helper Functions
# =============================

def obstacle_repulsion(pos):
    """
    Simple obstacle repulsion model.

    Returns:
        rep: repulsive vector away from nearby obstacles.
        min_clearance: minimum distance to obstacle boundary.
    """
    rep = np.zeros(2)
    min_clearance = 10.0

    for ox, oy, r in obstacles:
        obs = np.array([ox, oy])
        diff = pos - obs
        dist = np.linalg.norm(diff) + 1e-6
        clearance = dist - r
        min_clearance = min(min_clearance, clearance)

        if dist < r + 2.8:
            rep += (diff / dist) * (1.0 / max(clearance, 0.15)) * 0.65

    return rep, min_clearance


def perception_score(pos):
    """
    Perception score proxy.

    Higher when the UAV is farther from obstacles and has more visible free space.
    """
    _, clearance = obstacle_repulsion(pos)

    clearance_score = np.clip(clearance / 4.0, 0, 1)

    # Penalize central cluttered region
    corridor_penalty = 0.15 * np.exp(
        -np.linalg.norm(pos - np.array([10, 10]))**2 / 30
    )

    return float(
        np.clip(
            clearance_score - corridor_penalty + np.random.normal(0, 0.02),
            0,
            1
        )
    )


def communication_score(pos, centroid):
    """
    Communication score proxy.

    Higher when the UAV is closer to the swarm centroid.
    """
    dist = np.linalg.norm(pos - centroid)

    return float(
        np.clip(
            1 - dist / 6.0 + np.random.normal(0, 0.02),
            0,
            1
        )
    )


def task_score(pos):
    """
    Task score proxy.

    Higher when the UAV has made more progress toward the goal.
    """
    d_goal = np.linalg.norm(goal - pos)

    return float(
        np.clip(
            1 - d_goal / np.linalg.norm(goal - start) + np.random.normal(0, 0.02),
            0,
            1
        )
    )

print("Helper functions loaded.")


In [ ]:

# =============================
# 2. Strategy Simulation
# =============================

def simulate_strategy(strategy, wp=0.62, wc=0.23, wg=0.15, delta=0.08):
    """
    Simulate one leader-selection strategy in the same obstacle environment.

    Args:
        strategy: one of ["static", "task", "communication", "perception"].
        wp, wc, wg: perception, communication, and geometry weights.
        delta: leader-switching threshold.

    Returns:
        dict containing trajectories, leader history, and evaluation metrics.
    """
    positions = np.array(
        [start + offset for offset in formation_offsets],
        dtype=float
    )

    trajectories = [[p.copy()] for p in positions]

    leader_history = []
    current_leader = 0
    switch_count = 0

    for t in range(steps):

        centroid = positions.mean(axis=0)

        scores = []

        for i, p in enumerate(positions):

            P = perception_score(p)
            C = communication_score(p, centroid)
            G = np.clip(1 - np.linalg.norm(p - centroid) / 5, 0, 1)
            T = task_score(p)

            if strategy == "static":
                score = 1.0 if i == 0 else 0.0

            elif strategy == "task":
                score = 0.75 * T + 0.15 * G + 0.10 * C

            elif strategy == "communication":
                score = 0.75 * C + 0.15 * G + 0.10 * T

            elif strategy == "perception":
                score = wp * P + wc * C + wg * G

            else:
                raise ValueError("Unknown strategy.")

            scores.append(score)

        candidate = int(np.argmax(scores))

        if strategy == "static":
            leader = 0

        else:
            # Hysteresis-like threshold.
            # Switch only if candidate is sufficiently better than current leader.
            if scores[candidate] - scores[current_leader] > delta:
                leader = candidate
            else:
                leader = current_leader

        if leader != current_leader:
            switch_count += 1

        current_leader = leader
        leader_history.append(leader)

        # Leader moves toward goal while avoiding obstacles.
        leader_pos = positions[leader]

        direction = goal - leader_pos
        direction = direction / (np.linalg.norm(direction) + 1e-6)

        rep, _ = obstacle_repulsion(leader_pos)

        # Different strategies use slightly different obstacle responsiveness.
        if strategy == "perception":
            move = 0.19 * direction + 0.18 * rep

        elif strategy == "communication":
            move = 0.18 * direction + 0.12 * rep

        elif strategy == "task":
            move = 0.21 * direction + 0.09 * rep

        else:
            move = 0.18 * direction + 0.08 * rep

        leader_new = leader_pos + move

        # Keep formation around current leader reference frame.
        desired_positions = np.zeros_like(positions)

        for i in range(n_uavs):
            desired_positions[i] = (
                leader_new + (formation_offsets[i] - formation_offsets[leader])
            )

        # Followers move toward desired formation positions.
        new_positions = positions.copy()

        for i in range(n_uavs):

            if i == leader:
                new_positions[i] = leader_new

            else:
                follow_vec = desired_positions[i] - positions[i]
                rep_i, _ = obstacle_repulsion(positions[i])
                new_positions[i] = positions[i] + 0.18 * follow_vec + 0.10 * rep_i

        positions = np.clip(new_positions, 0.5, world_size - 0.5)

        for i in range(n_uavs):
            trajectories[i].append(positions[i].copy())

    trajectories = [np.array(traj) for traj in trajectories]

    # -----------------------------
    # Metrics
    # -----------------------------
    min_clearances = []
    form_errors = []

    for t in range(len(trajectories[0])):

        pos_t = np.array(
            [trajectories[i][t] for i in range(n_uavs)]
        )

        for p in pos_t:
            _, clearance = obstacle_repulsion(p)
            min_clearances.append(clearance)

        rel = pos_t - pos_t[0]
        target_rel = formation_offsets - formation_offsets[0]

        form_errors.append(
            np.mean(
                np.linalg.norm(rel - target_rel, axis=1)
            )
        )

    final_positions = np.array(
        [trajectories[i][-1] for i in range(n_uavs)]
    )

    final_dist = np.linalg.norm(
        final_positions.mean(axis=0) - goal
    )

    collision_risk = float(
        np.mean(np.array(min_clearances) < 0.4)
    )

    formation_error = float(np.mean(form_errors))

    success_proxy = float(
        np.clip(1 - final_dist / 20 - collision_risk, 0, 1)
    )

    robustness = float(
        success_proxy
        - 0.25 * collision_risk
        - 0.05 * switch_count / steps
        - 0.08 * formation_error
    )

    return {
        "strategy": strategy,
        "trajectories": trajectories,
        "leader_history": leader_history,
        "switch_count": switch_count,
        "collision_risk": collision_risk,
        "formation_error": formation_error,
        "success_proxy": success_proxy,
        "robustness": robustness
    }

print("Simulation function loaded.")


In [ ]:

# =============================
# 3. Run All Strategies
# =============================

results = {
    s: simulate_strategy(s)
    for s in strategies
}

metrics_df = pd.DataFrame([
    {
        "strategy": strategy_titles[s],
        "switch_count": results[s]["switch_count"],
        "collision_risk": results[s]["collision_risk"],
        "formation_error": results[s]["formation_error"],
        "success_proxy": results[s]["success_proxy"],
        "robustness": results[s]["robustness"]
    }
    for s in strategies
])

metrics_df


In [ ]:

# =============================
# 4. Qualitative Trajectory Visualization
# =============================

fig, axes = plt.subplots(2, 2, figsize=(12, 11))
axes = axes.ravel()

for ax, s in zip(axes, strategies):

    res = results[s]

    ax.set_title(strategy_titles[s])
    ax.set_xlim(0, world_size)
    ax.set_ylim(0, world_size)
    ax.set_aspect("equal")

    # Draw obstacles.
    for ox, oy, r in obstacles:
        ax.add_patch(Circle((ox, oy), r, alpha=0.25))
        ax.add_patch(Circle((ox, oy), r + 0.6, alpha=0.08))

    # Start and goal markers.
    ax.scatter(start[0], start[1], marker="s", s=80, label="Start")
    ax.scatter(goal[0], goal[1], marker="*", s=160, label="Goal")

    # UAV trajectories.
    for i, traj in enumerate(res["trajectories"]):
        ax.plot(traj[:, 0], traj[:, 1], linewidth=1.5, alpha=0.85)
        ax.scatter(traj[-1, 0], traj[-1, 1], s=35)

    # Leader path.
    leader_pts = []

    for t, leader in enumerate(res["leader_history"]):
        leader_pts.append(res["trajectories"][leader][t])

    leader_pts = np.array(leader_pts)

    ax.plot(
        leader_pts[:, 0],
        leader_pts[:, 1],
        linewidth=3.0,
        linestyle="--",
        label="Leader path"
    )

    ax.text(
        0.6,
        19.0,
        f"Switches: {res['switch_count']}\n"
        f"Form err: {res['formation_error']:.2f}\n"
        f"Risk: {res['collision_risk']:.2f}",
        fontsize=9,
        bbox=dict(boxstyle="round", alpha=0.15)
    )

    ax.grid(True, alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=3
)

fig.suptitle(
    "UAV Formation Trajectories under Different Leader Selection Strategies",
    fontsize=15
)

plt.tight_layout(rect=[0, 0.04, 1, 0.96])

combined_path = out_dir / "uav_formation_strategy_comparison.png"

plt.savefig(combined_path, dpi=300)

plt.show()

print("Saved:", combined_path)


In [ ]:

# =============================
# 5. Quantitative Metric Visualization
# =============================

metrics_path = out_dir / "uav_strategy_visual_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

print("Saved metric CSV:", metrics_path)

metric_paths = []

for metric in [
    "switch_count",
    "collision_risk",
    "formation_error",
    "success_proxy",
    "robustness"
]:

    plt.figure(figsize=(7, 4))

    plt.bar(
        metrics_df["strategy"],
        metrics_df[metric]
    )

    plt.xticks(rotation=20, ha="right")
    plt.ylabel(metric)

    plt.title(f"Visual Simulation Metric: {metric}")

    plt.tight_layout()

    p = out_dir / f"metric_{metric}.png"

    plt.savefig(p, dpi=300)

    metric_paths.append(p)

    plt.show()

print("Saved metric figures:")
for p in metric_paths:
    print(p)



# Interpretation Notes

This visualization compares four leader-selection strategies under the same complex obstacle environment:

1. Static Leader
2. Task-based Leader
3. Communication-based Leader
4. Perception-aware Leader

The dashed line represents the leader path. Solid lines represent UAV trajectories.

Key interpretation points:

- Static leader usually has stable switching behavior but limited environmental adaptability.
- Task-based leader tends to follow goal progress aggressively, which can increase obstacle exposure.
- Communication-based leader tends to preserve formation more effectively but may not always choose the best perceptual viewpoint.
- Perception-aware leader tends to adapt more actively to obstacle distribution, often reducing collision risk, but may increase switching frequency.

This figure is intended as a qualitative visualization, not as a high-fidelity physical UAV simulation.



# 6. Dynamic Animation of UAV Formation Trajectories

This section creates dynamic animations for the UAV formation under different leader-selection strategies.

It assumes that the previous cells have already generated:

- `results`
- `strategies`
- `strategy_titles`
- `obstacles`
- `start`
- `goal`
- `world_size`
- `out_dir`

The animation shows UAV positions, trajectory trails, the active leader UAV, the leader path, obstacles, and the goal position.

The outputs will be saved into:

`figure_visualization/animations/`


In [ ]:

# =============================
# 6.1 Animation Imports
# =============================

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

animation_dir = out_dir / "animations"
animation_dir.mkdir(exist_ok=True)

print("Animation folder:", animation_dir)


In [ ]:

# =============================
# 6.2 Single-Strategy Animation Function
# =============================

def animate_single_strategy(strategy, interval=120, save_html=True, save_gif=False):
    """
    Create an animation for one leader-selection strategy.

    Args:
        strategy: one of ["static", "task", "communication", "perception"].
        interval: delay between frames in milliseconds.
        save_html: save interactive HTML animation.
        save_gif: try saving GIF. Requires pillow.

    Returns:
        anim: matplotlib animation object.
    """

    res = results[strategy]
    trajectories = res["trajectories"]
    leader_history = res["leader_history"]

    n_frames = len(trajectories[0])

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.set_xlim(0, world_size)
    ax.set_ylim(0, world_size)
    ax.set_aspect("equal")
    ax.set_title(f"Dynamic UAV Formation: {strategy_titles[strategy]}")

    for ox, oy, r in obstacles:
        ax.add_patch(Circle((ox, oy), r, alpha=0.25))
        ax.add_patch(Circle((ox, oy), r + 0.6, alpha=0.08))

    ax.scatter(start[0], start[1], marker="s", s=80, label="Start")
    ax.scatter(goal[0], goal[1], marker="*", s=180, label="Goal")

    uav_points = [
        ax.plot([], [], marker="o", markersize=7, linestyle="None")[0]
        for _ in range(n_uavs)
    ]

    uav_trails = [
        ax.plot([], [], linewidth=1.5, alpha=0.75)[0]
        for _ in range(n_uavs)
    ]

    leader_point, = ax.plot([], [], marker="*", markersize=15, linestyle="None", label="Active Leader")
    leader_trail, = ax.plot([], [], linestyle="--", linewidth=2.5, label="Leader path")

    formation_lines = [
        ax.plot([], [], linewidth=1.0, alpha=0.4)[0]
        for _ in range(n_uavs - 1)
    ]

    info_text = ax.text(
        0.5,
        19.1,
        "",
        fontsize=9,
        bbox=dict(boxstyle="round", alpha=0.15)
    )

    ax.grid(True, alpha=0.25)
    ax.legend(loc="lower right")

    leader_path_x = []
    leader_path_y = []

    def init():
        for point in uav_points:
            point.set_data([], [])
        for trail in uav_trails:
            trail.set_data([], [])
        leader_point.set_data([], [])
        leader_trail.set_data([], [])
        for line in formation_lines:
            line.set_data([], [])
        info_text.set_text("")
        return uav_points + uav_trails + [leader_point, leader_trail, info_text] + formation_lines

    def update(frame):
        current_positions = []

        for i in range(n_uavs):
            traj = trajectories[i]
            x = traj[:frame+1, 0]
            y = traj[:frame+1, 1]

            uav_trails[i].set_data(x, y)
            uav_points[i].set_data([traj[frame, 0]], [traj[frame, 1]])
            current_positions.append(traj[frame])

        current_positions = np.array(current_positions)

        leader_id = leader_history[min(frame, len(leader_history)-1)]
        leader_pos = trajectories[leader_id][frame]

        leader_path_x.append(leader_pos[0])
        leader_path_y.append(leader_pos[1])

        leader_point.set_data([leader_pos[0]], [leader_pos[1]])
        leader_trail.set_data(leader_path_x, leader_path_y)

        idx = 0
        for i in range(n_uavs):
            if i == leader_id:
                continue

            formation_lines[idx].set_data(
                [leader_pos[0], current_positions[i, 0]],
                [leader_pos[1], current_positions[i, 1]]
            )
            idx += 1

        info_text.set_text(
            f"Frame: {frame}/{n_frames-1}\n"
            f"Leader ID: {leader_id}\n"
            f"Switches: {res['switch_count']}\n"
            f"Risk: {res['collision_risk']:.2f}\n"
            f"Formation error: {res['formation_error']:.2f}"
        )

        return uav_points + uav_trails + [leader_point, leader_trail, info_text] + formation_lines

    anim = FuncAnimation(
        fig,
        update,
        frames=n_frames,
        init_func=init,
        interval=interval,
        blit=True
    )

    html_path = animation_dir / f"{strategy}_uav_formation_animation.html"
    gif_path = animation_dir / f"{strategy}_uav_formation_animation.gif"

    if save_html:
        html_content = anim.to_jshtml()
        html_path.write_text(html_content, encoding="utf-8")
        print("Saved HTML:", html_path)

    if save_gif:
        try:
            anim.save(gif_path, writer="pillow", fps=8)
            print("Saved GIF:", gif_path)
        except Exception as e:
            print("GIF saving failed. HTML animation is still available.")
            print("Reason:", e)

    plt.close(fig)

    return anim


In [ ]:

# =============================
# 6.3 Generate One Animation for Inspection
# =============================

# Change the strategy name to inspect other methods:
# "static", "task", "communication", "perception"

anim = animate_single_strategy(
    strategy="perception",
    interval=120,
    save_html=True,
    save_gif=False
)

display(HTML(anim.to_jshtml()))


In [ ]:

# =============================
# 6.4 Generate HTML Animations for All Strategies
# =============================

all_animations = {}

for s in strategies:
    print("Generating animation for:", strategy_titles[s])
    all_animations[s] = animate_single_strategy(
        strategy=s,
        interval=120,
        save_html=True,
        save_gif=False
    )

print("All strategy animations have been generated.")


In [ ]:

# =============================
# 6.5 Combined 2x2 Dynamic Animation
# =============================

def animate_all_strategies(interval=120, save_html=True, save_gif=False):
    """
    Create a combined 2x2 animation comparing all four leader-selection strategies.
    """

    fig, axes = plt.subplots(2, 2, figsize=(12, 11))
    axes = axes.ravel()

    plot_objects = {}

    max_frames = min(
        len(results[s]["trajectories"][0])
        for s in strategies
    )

    for ax, s in zip(axes, strategies):

        ax.set_xlim(0, world_size)
        ax.set_ylim(0, world_size)
        ax.set_aspect("equal")
        ax.set_title(strategy_titles[s])

        for ox, oy, r in obstacles:
            ax.add_patch(Circle((ox, oy), r, alpha=0.25))
            ax.add_patch(Circle((ox, oy), r + 0.6, alpha=0.08))

        ax.scatter(start[0], start[1], marker="s", s=70)
        ax.scatter(goal[0], goal[1], marker="*", s=150)

        uav_points = [
            ax.plot([], [], marker="o", markersize=5, linestyle="None")[0]
            for _ in range(n_uavs)
        ]

        uav_trails = [
            ax.plot([], [], linewidth=1.1, alpha=0.7)[0]
            for _ in range(n_uavs)
        ]

        leader_point, = ax.plot([], [], marker="*", markersize=12, linestyle="None")
        leader_trail, = ax.plot([], [], linestyle="--", linewidth=2.0)

        info_text = ax.text(
            0.5,
            19.1,
            "",
            fontsize=8,
            bbox=dict(boxstyle="round", alpha=0.15)
        )

        ax.grid(True, alpha=0.25)

        plot_objects[s] = {
            "uav_points": uav_points,
            "uav_trails": uav_trails,
            "leader_point": leader_point,
            "leader_trail": leader_trail,
            "info_text": info_text,
            "leader_x": [],
            "leader_y": []
        }

    fig.suptitle(
        "Dynamic UAV Formation Comparison under Different Leader Selection Strategies",
        fontsize=15
    )

    def init():
        artists = []

        for s in strategies:
            objs = plot_objects[s]

            for point in objs["uav_points"]:
                point.set_data([], [])
                artists.append(point)

            for trail in objs["uav_trails"]:
                trail.set_data([], [])
                artists.append(trail)

            objs["leader_point"].set_data([], [])
            objs["leader_trail"].set_data([], [])
            objs["info_text"].set_text("")

            artists.extend([
                objs["leader_point"],
                objs["leader_trail"],
                objs["info_text"]
            ])

        return artists

    def update(frame):
        artists = []

        for s in strategies:

            res = results[s]
            objs = plot_objects[s]

            trajectories = res["trajectories"]
            leader_history = res["leader_history"]

            for i in range(n_uavs):
                traj = trajectories[i]

                objs["uav_trails"][i].set_data(
                    traj[:frame+1, 0],
                    traj[:frame+1, 1]
                )

                objs["uav_points"][i].set_data(
                    [traj[frame, 0]],
                    [traj[frame, 1]]
                )

                artists.append(objs["uav_trails"][i])
                artists.append(objs["uav_points"][i])

            leader_id = leader_history[min(frame, len(leader_history)-1)]
            leader_pos = trajectories[leader_id][frame]

            objs["leader_x"].append(leader_pos[0])
            objs["leader_y"].append(leader_pos[1])

            objs["leader_point"].set_data(
                [leader_pos[0]],
                [leader_pos[1]]
            )

            objs["leader_trail"].set_data(
                objs["leader_x"],
                objs["leader_y"]
            )

            objs["info_text"].set_text(
                f"Frame: {frame}\n"
                f"Leader: {leader_id}\n"
                f"Switches: {res['switch_count']}"
            )

            artists.extend([
                objs["leader_point"],
                objs["leader_trail"],
                objs["info_text"]
            ])

        return artists

    anim = FuncAnimation(
        fig,
        update,
        frames=max_frames,
        init_func=init,
        interval=interval,
        blit=True
    )

    html_path = animation_dir / "all_strategies_dynamic_comparison.html"
    gif_path = animation_dir / "all_strategies_dynamic_comparison.gif"

    if save_html:
        html_path.write_text(anim.to_jshtml(), encoding="utf-8")
        print("Saved HTML:", html_path)

    if save_gif:
        try:
            anim.save(gif_path, writer="pillow", fps=8)
            print("Saved GIF:", gif_path)
        except Exception as e:
            print("GIF saving failed. HTML animation is still available.")
            print("Reason:", e)

    plt.close(fig)

    return anim


combined_anim = animate_all_strategies(
    interval=120,
    save_html=True,
    save_gif=False
)

display(HTML(combined_anim.to_jshtml()))


In [ ]:

# =============================
# 6.6 Optional: Save GIF Files
# =============================

# This step may take longer.
# It requires pillow. If pillow is missing, install it with:
# pip install pillow

# Uncomment the following lines if you want GIF outputs.

# for s in strategies:
#     animate_single_strategy(
#         strategy=s,
#         interval=120,
#         save_html=False,
#         save_gif=True
#     )

# animate_all_strategies(
#     interval=120,
#     save_html=False,
#     save_gif=True
# )
